# Fridge to Recipe

### Import libraries and dataset

In [2]:
import pandas as pd
df = pd.read_csv('./data/recipes_data.csv')

print(df.shape)
df.head()


(2231142, 7)


,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


Parse NER-column to list

In [3]:
import ast
df['NER_parsed'] = df['NER'].apply(ast.literal_eval)

Transform list to strings for TF-IDF

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
df['NER_text'] = df['NER_parsed'].apply(lambda x: ' '.join(x))


Fit TF-IDF on database

In [7]:
tfidf_matching = TfidfVectorizer()
tfidf_matrix_matching = tfidf_matching.fit_transform(df['NER_text'])

### TF-IDF Cosine similarity matching


In [8]:
user_input = "chicken garlic rice"
user_vector = tfidf_matching.transform([user_input])

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(user_vector, tfidf_matrix_matching)

Sort the scores

In [10]:
import numpy as np

scores = similarities[0]
sorted_indices = scores.argsort()
sorted_indices_desc = sorted_indices[::-1]
top_indices = sorted_indices_desc[:5]
print(sorted_indices_desc)

[ 256530  238459 1825610 ... 1043940 1870636 1115570]


In [11]:
top_recipes = df.iloc[top_indices]
print(top_recipes[['title', 'NER_text']])

                                                     title      NER_text
256530                                    Chicken And Rice  rice chicken
238459                                   Pileau(Pearl O)    rice chicken
1825610  CHEEZ WHIZ White Cheddar Chicken & Broccoli Su...  rice chicken
665165                                    Chicken And Rice  rice chicken
1843760             CHEEZ WHIZ Chicken and Broccoli Dinner  rice chicken


In [12]:
top_scores = similarities[0][top_indices]
for idx, score in zip(top_indices, top_scores):
    print(f"{df.iloc[idx]['title']} — similarity: {score:.3f}")

Chicken And Rice — similarity: 0.888
Pileau(Pearl O)   — similarity: 0.888
CHEEZ WHIZ White Cheddar Chicken & Broccoli Supper — similarity: 0.888
Chicken And Rice — similarity: 0.888
CHEEZ WHIZ Chicken and Broccoli Dinner — similarity: 0.888


# Putting it all together

In [1]:
def find_matching_recipes(user_ingredients, top_n=5):
    user_vector = tfidf_matching.transform([user_ingredients])
    similarities = cosine_similarity(user_vector, tfidf_matrix_matching)
    top_indices = similarities[0].argsort()[::-1][:top_n]
    top_scores = similarities[0][top_indices]
    
    results = df.iloc[top_indices][['title', 'NER_text']].copy()
    results['similarity_score'] = top_scores
    return results